# Panel de viaje · Sesión Overpass · Pruebas pendientes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tragabytes/panel-viaje/blob/main/tests/fase1_overpass.ipynb)

**Proyecto:** https://github.com/tragabytes/panel-viaje

Sesión específica de Overpass. Cubre las tres tareas que la sesión 05 dejó pendientes por saturación de mirrors:

1. **Prueba 0** — salud de los mirrors. Si todos están caídos otra vez, se aborta limpio.
2. **Prueba 5 (bloque 1.4)** — pueblos cercanos a un punto GPS. Crítica: alimenta la vista 3.
3. **Prueba 3 (bloque 1.4)** — Overpass como fuente de POIs vs Wikidata.
4. **Próxima salida de autovía (bloque 1.2, segunda parte)** — `motorway_junction` cercanos filtrados por heading + validación opción C (consulta única + filtrado local).

Formato compacto. Cada prueba es independiente tras el setup.

## Setup

In [ ]:
import requests, time, math

UA = "panel-viaje-tragabytes/0.1 (https://github.com/tragabytes/panel-viaje; contacto via GitHub)"
HEADERS_OV = {"User-Agent": UA}

MIRRORS = [
    ("overpass-api.de",       "https://overpass-api.de/api/interpreter"),
    ("kumi.systems",          "https://overpass.kumi.systems/api/interpreter"),
    ("private.coffee",        "https://overpass.private.coffee/api/interpreter"),
]

OVERPASS_OK = None

def fetch_overpass(query, endpoint=None, max_intentos=4, timeout=60):
    """Reintentos espaciados; mismo patrón validado en sesión 05."""
    url = endpoint or OVERPASS_OK
    if not url:
        return {"ok": False, "error": "sin endpoint Overpass disponible", "intentos": 0}
    esperas = [0, 5, 15, 30]
    last_err = None
    for intento in range(max_intentos):
        if esperas[intento]: time.sleep(esperas[intento])
        t0 = time.time()
        try:
            r = requests.post(url, data={"data": query}, headers=HEADERS_OV, timeout=timeout)
            dt = (time.time() - t0) * 1000
            if r.status_code == 200:
                return {"ok": True, "data": r.json(), "ms": dt, "bytes": len(r.content), "intentos": intento + 1}
            last_err = f"HTTP {r.status_code}: {r.text[:120]}"
        except Exception as e:
            last_err = f"{type(e).__name__}: {str(e)[:120]}"
    return {"ok": False, "error": last_err, "intentos": max_intentos}

def distancia_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    """Rumbo en grados desde el punto 1 hacia el punto 2 (0=N, 90=E)."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    y = math.sin(dl) * math.cos(p2)
    x = math.cos(p1)*math.sin(p2) - math.sin(p1)*math.cos(p2)*math.cos(dl)
    return (math.degrees(math.atan2(y, x)) + 360) % 360

def diff_angular(a, b):
    """Diferencia angular mínima entre dos rumbos en grados (0..180)."""
    d = abs(a - b) % 360
    return d if d <= 180 else 360 - d

def avanzar_punto(lat, lon, heading, distancia_km):
    """Avanza un punto en la dirección del heading una distancia dada en km."""
    R = 6371
    d = distancia_km / R
    h = math.radians(heading)
    p1 = math.radians(lat)
    l1 = math.radians(lon)
    p2 = math.asin(math.sin(p1)*math.cos(d) + math.cos(p1)*math.sin(d)*math.cos(h))
    l2 = l1 + math.atan2(math.sin(h)*math.sin(d)*math.cos(p1), math.cos(d) - math.sin(p1)*math.sin(p2))
    return math.degrees(p2), math.degrees(l2)

print("Setup OK")

## Prueba 0 — Salud de los mirrors

Query mínima a cada uno. El primero que responda con 200 será `OVERPASS_OK` para el resto del notebook. Si los tres fallan, las celdas siguientes lo detectan y abortan.

**Resultado en sesión 06 (7 abr 2026):** 1/3 mirrors operativos (overpass-api.de). Latencia anómalamente alta (8 s) para query trivial, indicando mirror operativo pero cargado. Confirmada en 2/2 sesiones la inestabilidad estructural de Overpass.

In [ ]:
QUERY_PING = '[out:json][timeout:10];out count;'

resultados_ping = []
for nombre, url in MIRRORS:
    t0 = time.time()
    try:
        r = requests.post(url, data={"data": QUERY_PING}, headers=HEADERS_OV, timeout=15)
        dt = (time.time() - t0) * 1000
        ok = r.status_code == 200
        msg = f"HTTP {r.status_code}" if not ok else f"OK · {dt:.0f} ms"
    except Exception as e:
        ok = False
        msg = f"{type(e).__name__}: {str(e)[:60]}"
    resultados_ping.append((nombre, url, ok, msg))
    estado = "✓" if ok else "✗"
    print(f"  {estado} {nombre:20s}  {msg}")

vivos = [(n, u) for n, u, ok, _ in resultados_ping if ok]
if vivos:
    OVERPASS_OK = vivos[0][1]
    print(f"\nMirror elegido: {vivos[0][0]}  ({len(vivos)}/3 disponibles)")
else:
    print("\n⚠ Los tres mirrors están caídos. Las celdas siguientes abortarán limpiamente.")
    print("  Acción: reintentar la sesión más tarde.")

## Prueba 5 — Pueblos cercanos a un punto GPS

La más crítica: sin esto no hay vista 3. Radio 15 km, `place=village|town|city`, top 4 por distancia.

3 puntos en carretera real (no sobre pueblo) para simular el caso del coche circulando.

**Bug encontrado y corregido en sesión 06:** la primera versión usaba `out tags;`, que devuelve nodos solo con tags, **sin coordenadas**. Las distancias salían absurdas (~4500 km, contra el punto (0,0) en el Golfo de Guinea). Corregido a `out;`.

In [ ]:
PUNTOS_CARRETERA = [
    {"nombre": "A-2 cerca de Medinaceli", "lat": 41.1856, "lon": -2.4570},
    {"nombre": "A-5 cerca de Trujillo",   "lat": 39.4760, "lon": -5.8200},
    {"nombre": "N-VI cerca de Astorga",   "lat": 42.4420, "lon": -6.0800},
]

def query_pueblos_cercanos(lat, lon, radio_m=15000):
    # IMPORTANTE: 'out;' (no 'out tags;'), si no, los nodos vienen sin lat/lon.
    return f'''[out:json][timeout:50];
(
  node["place"~"^(village|town|city)$"](around:{radio_m},{lat},{lon});
);
out;'''

resultados_cercanos = {}

if not OVERPASS_OK:
    print("⚠ Sin mirror activo. Saltando prueba 5.")
else:
    for p in PUNTOS_CARRETERA:
        print(f"\n=== {p['nombre']} ({p['lat']}, {p['lon']}) ===")
        res = fetch_overpass(query_pueblos_cercanos(p['lat'], p['lon'], 15000))
        if not res['ok']:
            print(f"  ERROR: {res['error'][:120]}")
            resultados_cercanos[p['nombre']] = {"ok": False, "error": res['error']}
            continue
        elementos = res['data'].get('elements', [])
        pueblos = []
        sin_coord = 0
        for e in elementos:
            tags = e.get('tags', {})
            name = tags.get('name')
            if not name: continue
            elat, elon = e.get('lat'), e.get('lon')
            if elat is None or elon is None:
                sin_coord += 1
                continue
            d = distancia_m(p['lat'], p['lon'], elat, elon)
            pueblos.append({"name": name, "place": tags.get('place'), "dist": d, "pop": tags.get('population')})
        pueblos.sort(key=lambda x: x['dist'])
        sin_nombre = sum(1 for el in elementos if not el.get('tags', {}).get('name'))
        extra = f" · {sin_coord} sin coord" if sin_coord else ""
        print(f"  {len(pueblos)} con nombre · {sin_nombre} sin nombre{extra} · {res['ms']:.0f} ms · {res['bytes']} B · {res['intentos']} intento(s)")
        print("  Top 10 por distancia:")
        for pu in pueblos[:10]:
            pop = f"pop={pu['pop']}" if pu['pop'] else ""
            print(f"    {pu['dist']/1000:5.1f} km  {pu['name'][:32]:32s} [{pu['place']}] {pop}")
        resultados_cercanos[p['nombre']] = {
            "ok": True, "ms": res['ms'], "bytes": res['bytes'],
            "total": len(pueblos), "top4": pueblos[:4],
        }
        time.sleep(3)

## Prueba 3 — Overpass como fuente de POIs vs Wikidata

Mismos 5 municipios de la sesión 05. Radio 1500 m. Tags: `historic=*`, `tourism=viewpoint|attraction`, `natural=peak`, `building=castle|church|monastery|cathedral|chapel`.

Referencia Wikidata sesión 05 (POIs / con imagen): Medinaceli 7/7, Albarracín 28/19, Chinchón 12/12, Trujillo 21/14, Pedraza 3/3.

**Ajustes en sesión 06 tras 504 inicial en Medinaceli:**
- `[timeout:180]` interno (servidor)
- `timeout=120` cliente Python
- Radio bajado de 2000 a 1500 m
- Pausa entre municipios subida a 5 s

**Resultado clave:** Pedraza pasó de 3 a 15 POIs con nombre. Trujillo de 21 a 37. Overpass resulta ser **mejor** que Wikidata para inventario de POIs en pueblos pequeños. Lleva al cambio de la decisión 11 (Overpass deja de ser fallback y pasa a ser fuente principal de inventario).

In [ ]:
MUNICIPIOS_POI = [
    {"nombre": "Medinaceli", "lat": 41.1731, "lon": -2.4344, "wd_pois": 7,  "wd_img": 7},
    {"nombre": "Albarracín", "lat": 40.4064, "lon": -1.4433, "wd_pois": 28, "wd_img": 19},
    {"nombre": "Chinchón",   "lat": 40.1396, "lon": -3.4261, "wd_pois": 12, "wd_img": 12},
    {"nombre": "Trujillo",   "lat": 39.4600, "lon": -5.8825, "wd_pois": 21, "wd_img": 14},
    {"nombre": "Pedraza",    "lat": 41.1283, "lon": -3.8108, "wd_pois": 3,  "wd_img": 3},
]

def query_pois_cercanos(lat, lon, radio_m=1500):
    return f'''[out:json][timeout:180];
(
  node["historic"](around:{radio_m},{lat},{lon});
  way["historic"](around:{radio_m},{lat},{lon});
  node["tourism"="viewpoint"](around:{radio_m},{lat},{lon});
  node["tourism"="attraction"](around:{radio_m},{lat},{lon});
  way["tourism"="attraction"](around:{radio_m},{lat},{lon});
  node["natural"="peak"](around:{radio_m},{lat},{lon});
  way["building"~"^(castle|church|monastery|cathedral|chapel)$"](around:{radio_m},{lat},{lon});
);
out center tags;'''

resultados_ov_pois = {}

if not OVERPASS_OK:
    print("⚠ Sin mirror activo. Saltando prueba 3.")
else:
    for m in MUNICIPIOS_POI:
        print(f"\n=== {m['nombre']} (Wikidata sesión 05: {m['wd_pois']} POIs, {m['wd_img']} con imagen) ===")
        res = fetch_overpass(query_pois_cercanos(m['lat'], m['lon'], 1500), timeout=120)
        if not res['ok']:
            print(f"  ERROR: {res['error'][:120]}")
            resultados_ov_pois[m['nombre']] = {"ok": False, "error": res['error']}
            time.sleep(5)
            continue
        elementos = res['data'].get('elements', [])
        con_nombre = [e for e in elementos if e.get('tags', {}).get('name')]
        print(f"  {len(elementos)} elem · {len(con_nombre)} con name · {res['ms']:.0f} ms · {res['bytes']} B · {res['intentos']} intento(s)")
        for e in con_nombre[:15]:
            tags = e.get('tags', {})
            name = tags.get('name', '?')
            tipo = ' '.join(f"{k}={tags[k]}" for k in ('historic','tourism','natural','building') if k in tags)
            print(f"    {name[:42]:42s}  [{tipo}]")
        if len(con_nombre) > 15:
            print(f"    ... y {len(con_nombre)-15} más")
        resultados_ov_pois[m['nombre']] = {
            "ok": True, "ms": res['ms'], "bytes": res['bytes'],
            "total": len(elementos), "con_nombre": len(con_nombre), "elementos": con_nombre,
        }
        time.sleep(5)

print("\n--- Comparativa Wikidata vs Overpass ---")
print(f"{'Municipio':14s} {'WD POIs':>9s} {'WD img':>8s} {'OV elem':>9s} {'OV nom':>8s}")
for m in MUNICIPIOS_POI:
    r = resultados_ov_pois.get(m['nombre'], {})
    if r.get('ok'):
        print(f"{m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d} {r['total']:>9d} {r['con_nombre']:>8d}")
    else:
        print(f"{m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d}    ERROR")

## Próxima salida de autovía (bloque 1.2, segunda parte)

**Estrategia validada en sesión 06:** consultar `node["highway"="motorway_junction"]` (no `way["highway"="motorway_link"]` como en el primer intento). Los `motorway_junction` son nodos únicos por salida que llevan `ref` (número/km) y a veces `name`. Los `motorway_link` son los ramales físicos, normalmente sin tags útiles.

Filtro geométrico: distancia al punto, rumbo desde el coche al junction, descartar si la diferencia angular con el heading es >45° (no está delante) o si la distancia es <300 m (ya pasada o demasiado cerca).

**Headings inventados** coherentes con la dirección normal de la vía:
- A-2 Medinaceli: ~60° (NE, dirección Zaragoza/Barcelona).
- A-5 Trujillo: ~250° (WSW, dirección Mérida/Badajoz).
- N-VI Astorga: ~290° (WNW, dirección Galicia).

In [ ]:
PUNTOS_AUTOVIA = [
    {"nombre": "A-2 cerca de Medinaceli", "lat": 41.1856, "lon": -2.4570, "heading": 60,  "sentido": "NE → Zaragoza"},
    {"nombre": "A-5 cerca de Trujillo",   "lat": 39.4760, "lon": -5.8200, "heading": 250, "sentido": "WSW → Mérida"},
    {"nombre": "N-VI cerca de Astorga",   "lat": 42.4420, "lon": -6.0800, "heading": 290, "sentido": "WNW → Galicia"},
]

def query_motorway_junctions(lat, lon, radio_m=10000):
    return f'''[out:json][timeout:120];
(
  node["highway"="motorway_junction"](around:{radio_m},{lat},{lon});
);
out;'''

def analizar_junctions(punto, elementos, dist_min_m=300, tolerancia_deg=45):
    """Filtra junctions delante del coche en la dirección de marcha."""
    candidatos = []
    for e in elementos:
        elat, elon = e.get('lat'), e.get('lon')
        if elat is None or elon is None: continue
        d = distancia_m(punto['lat'], punto['lon'], elat, elon)
        if d < dist_min_m: continue
        rumbo_a = bearing_deg(punto['lat'], punto['lon'], elat, elon)
        diff = diff_angular(rumbo_a, punto['heading'])
        if diff > tolerancia_deg: continue
        tags = e.get('tags', {})
        candidatos.append({
            "id": e.get('id'),
            "dist": d,
            "rumbo_a": rumbo_a,
            "diff_heading": diff,
            "ref":  tags.get('ref'),
            "name": tags.get('name'),
        })
    return sorted(candidatos, key=lambda x: x['dist'])

resultados_salidas = {}

if not OVERPASS_OK:
    print("⚠ Sin mirror activo. Saltando prueba de próxima salida.")
else:
    for p in PUNTOS_AUTOVIA:
        print(f"\n=== {p['nombre']}  heading={p['heading']}° ({p['sentido']}) ===")
        res = fetch_overpass(query_motorway_junctions(p['lat'], p['lon'], 10000), timeout=120)
        if not res['ok']:
            print(f"  ERROR: {res['error'][:120]}")
            resultados_salidas[p['nombre']] = {"ok": False, "error": res['error']}
            time.sleep(5)
            continue
        elementos = res['data'].get('elements', [])
        candidatos = analizar_junctions(p, elementos)
        print(f"  {len(elementos)} motorway_junction en 10 km · {len(candidatos)} delante (Δ<45°, dist>300m) · {res['ms']:.0f} ms · {res['bytes']} B")
        if not candidatos:
            print("  ⚠ ninguna salida cumple. Mostrando los 5 junctions más cercanos sin filtrar:")
            sin_filtro = []
            for e in elementos:
                elat, elon = e.get('lat'), e.get('lon')
                if elat is None: continue
                d = distancia_m(p['lat'], p['lon'], elat, elon)
                rumbo_a = bearing_deg(p['lat'], p['lon'], elat, elon)
                diff = diff_angular(rumbo_a, p['heading'])
                tags = e.get('tags', {})
                sin_filtro.append({"dist": d, "diff": diff, "ref": tags.get('ref'), "name": tags.get('name')})
            sin_filtro.sort(key=lambda x: x['dist'])
            for c in sin_filtro[:5]:
                etiqueta = f"ref={c['ref']}" if c['ref'] else "(sin ref)"
                if c['name']: etiqueta += f"  name={c['name']}"
                print(f"    {c['dist']/1000:5.2f} km  Δ={c['diff']:5.1f}°  {etiqueta}")
        else:
            print("  Próximas salidas:")
            for c in candidatos[:5]:
                etiqueta = f"ref={c['ref']}" if c['ref'] else "(sin ref)"
                if c['name']: etiqueta += f"  name={c['name']}"
                print(f"    {c['dist']/1000:5.2f} km  Δheading={c['diff_heading']:5.1f}°  {etiqueta}")
        resultados_salidas[p['nombre']] = {
            "ok": True, "ms": res['ms'], "bytes": res['bytes'],
            "total": len(elementos), "delante": len(candidatos),
            "top": candidatos[:3],
        }
        time.sleep(5)

## Validación opción C — consulta única (50 km) + filtrado local

**Por qué esta celda existe.** A velocidad de coche, la información de "próxima salida" tiene que actualizarse en pantalla cada pocos segundos para ser honesta. Refrescar contra Overpass cada actualización significaría 100+ peticiones por viaje, contra el principio de minimizar consumo de datos y batería.

**Estrategia opción C:** una sola consulta amplia (radio 50 km) cuando el coche entra en autovía nueva. La lista de junctions se cachea en memoria. Cada actualización del GPS recalcula localmente la próxima salida con el filtro de heading + distancia mínima, sin volver a tocar Overpass. Cuando el coche se acerca al final del segmento cacheado, se pide el siguiente.

**Lo que valida esta celda:**
1. Que una sola consulta de 50 km es asumible (tamaño y latencia).
2. Que el filtrado local sobre datos cacheados funciona al avanzar el coche, simulando puntos a +5 km, +10 km, +20 km y +35 km en la dirección del heading.

**Resultado en sesión 06:** validada como arquitectura. Total cacheado <25 KB por consulta, cobertura 50 km = ~25 min a 120 km/h, estimado 3-5 consultas Overpass por viaje de 3 h. La A-6 (Astorga) muestra refs monotónicas (333, 347, 360, 366, 376) confirmando mecánica correcta. La A-2 muestra degradación del filtro angular cuando el coche avanza con heading fijo, lo cual NO ocurre en producción porque el GPS proporciona heading actualizado en cada cálculo.

In [ ]:
def query_junctions_radio_amplio(lat, lon, radio_m=50000):
    return f'''[out:json][timeout:120];
(
  node["highway"="motorway_junction"](around:{radio_m},{lat},{lon});
);
out;'''

def filtrar_proxima_salida(lat, lon, heading, junctions, dist_min_m=300, tolerancia_deg=45):
    """Filtra y ordena los junctions cacheados para encontrar la próxima salida."""
    candidatos = []
    for j in junctions:
        elat, elon = j.get('lat'), j.get('lon')
        if elat is None: continue
        d = distancia_m(lat, lon, elat, elon)
        if d < dist_min_m: continue
        rumbo_a = bearing_deg(lat, lon, elat, elon)
        diff = diff_angular(rumbo_a, heading)
        if diff > tolerancia_deg: continue
        tags = j.get('tags', {})
        candidatos.append({"dist": d, "diff": diff, "ref": tags.get('ref'), "name": tags.get('name')})
    return sorted(candidatos, key=lambda x: x['dist'])

cache_junctions = {}
resultados_opcion_c = {}

if not OVERPASS_OK:
    print("⚠ Sin mirror activo. Saltando validación opción C.")
else:
    for p in PUNTOS_AUTOVIA:
        print(f"\n=== {p['nombre']}  heading={p['heading']}° ===")
        res = fetch_overpass(query_junctions_radio_amplio(p['lat'], p['lon'], 50000), timeout=120)
        if not res['ok']:
            print(f"  ERROR: {res['error'][:120]}")
            resultados_opcion_c[p['nombre']] = {"ok": False, "error": res['error']}
            time.sleep(5)
            continue
        junctions = res['data'].get('elements', [])
        cache_junctions[p['nombre']] = junctions
        print(f"  PETICIÓN ÚNICA: {len(junctions)} junctions en 50 km · {res['ms']:.0f} ms · {res['bytes']} B ({res['bytes']/1024:.1f} KB)")
        simulacion = []
        for km_avanzados in [0, 5, 10, 20, 35]:
            if km_avanzados == 0:
                lat_sim, lon_sim = p['lat'], p['lon']
            else:
                lat_sim, lon_sim = avanzar_punto(p['lat'], p['lon'], p['heading'], km_avanzados)
            candidatos = filtrar_proxima_salida(lat_sim, lon_sim, p['heading'], junctions)
            if candidatos:
                c = candidatos[0]
                etiqueta = f"ref={c['ref']}" if c['ref'] else "(sin ref)"
                if c['name']: etiqueta += f"  name={c['name']}"
                marca = "📍" if km_avanzados == 0 else f"+{km_avanzados:2d}km"
                print(f"  {marca}  próxima salida → {c['dist']/1000:5.2f} km  Δ={c['diff']:5.1f}°  {etiqueta}")
                simulacion.append({"km": km_avanzados, "dist": c['dist'], "ref": c['ref']})
            else:
                marca = "📍" if km_avanzados == 0 else f"+{km_avanzados:2d}km"
                print(f"  {marca}  ninguna salida en cache cumple los filtros")
                simulacion.append({"km": km_avanzados, "dist": None, "ref": None})
        resultados_opcion_c[p['nombre']] = {
            "ok": True, "ms": res['ms'], "bytes": res['bytes'],
            "junctions": len(junctions), "simulacion": simulacion,
        }
        time.sleep(5)

print("\n--- Veredicto opción C ---")
if cache_junctions:
    total_kb = sum(r['bytes'] for r in resultados_opcion_c.values() if r.get('ok')) / 1024
    print(f"  ✓ Datos cacheados en memoria total (3 puntos): ~{total_kb:.1f} KB")
    print(f"  ✓ Cobertura: 50 km por consulta = ~25 min de viaje a 120 km/h")
    print(f"  ✓ Cada filtrado local es instantáneo (sin red)")
    print(f"  ✓ Estimado para viaje de 3h en autovía: 3-5 consultas a Overpass")

## Resumen final

Celda que reúne todo en un informe compacto. Pegar al chat para análisis y actualización del seguimiento.

**Bug corregido en sesión 06:** la primera versión leía `r['total_links']`, `r['delante']`, `r['salidas_unicas']`, claves de la versión inicial de la celda de próxima salida (motorway_link). Tras reescribir esa celda con motorway_junction, las claves cambiaron a `r['total']`, `r['delante']`, `r['top']`. El resumen se quedó desactualizado y reventaba con KeyError. Corregido.

In [ ]:
print("="*70)
print("RESUMEN SESIÓN OVERPASS")
print("="*70)

print("\n## P0 — Salud de mirrors")
for nombre, url, ok, msg in resultados_ping:
    print(f"  {'✓' if ok else '✗'} {nombre:20s}  {msg}")
print(f"  Mirror elegido: {OVERPASS_OK or 'NINGUNO'}")

print("\n## P5 — Pueblos cercanos a punto GPS (radio 15 km)")
if not resultados_cercanos:
    print("  (no ejecutada)")
for nombre, r in resultados_cercanos.items():
    if r.get('ok'):
        print(f"  {nombre:32s} {r['total']:3d} pueblos · {r['ms']:.0f} ms · {r['bytes']} B")
        for pu in r['top4']:
            print(f"    {pu['dist']/1000:5.1f} km  {pu['name']}")
    else:
        print(f"  {nombre:32s} ERROR")

print("\n## P3 — Overpass como fuente de POIs (radio 1500 m)")
if not resultados_ov_pois:
    print("  (no ejecutada)")
else:
    print(f"  {'Municipio':14s} {'WD POIs':>9s} {'WD img':>8s} {'OV elem':>9s} {'OV nom':>8s}")
    for m in MUNICIPIOS_POI:
        r = resultados_ov_pois.get(m['nombre'], {})
        if r.get('ok'):
            print(f"  {m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d} {r['total']:>9d} {r['con_nombre']:>8d}")
        else:
            print(f"  {m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d}    ERROR")

print("\n## Próxima salida — motorway_junction (radio 10 km, Δ<45°, dist>300m)")
if not resultados_salidas:
    print("  (no ejecutada)")
for nombre, r in resultados_salidas.items():
    if r.get('ok'):
        print(f"  {nombre:32s} {r['total']:3d} junctions · {r['delante']:3d} delante · {r['ms']:.0f} ms · {r['bytes']} B")
        for c in r['top']:
            etiqueta = f"ref={c['ref']}" if c['ref'] else "(sin ref)"
            if c.get('name'): etiqueta += f"  name={c['name']}"
            print(f"    {c['dist']/1000:5.2f} km  Δ={c['diff_heading']:5.1f}°  {etiqueta}")
    else:
        print(f"  {nombre:32s} ERROR")

print("\n## Opción C — consulta única (50 km) + filtrado local")
if not resultados_opcion_c:
    print("  (no ejecutada)")
for nombre, r in resultados_opcion_c.items():
    if r.get('ok'):
        print(f"  {nombre:32s} {r['junctions']:3d} junctions · {r['ms']:.0f} ms · {r['bytes']/1024:.1f} KB")
        for s in r['simulacion']:
            marca = "📍" if s['km'] == 0 else f"+{s['km']:2d}km"
            if s['dist'] is None:
                print(f"    {marca}  (sin salida)")
            else:
                print(f"    {marca}  {s['dist']/1000:5.2f} km  ref={s['ref']}")
    else:
        print(f"  {nombre:32s} ERROR")

print("\n" + "="*70)
print("Pega este resumen en el chat para análisis y actualización del seguimiento.")